In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
data = pd.read_csv('cows-detection-dataset.csv')

# First, let's see what's in the dataset
print("Dataset columns:", data.columns.tolist())
print("\nFirst few rows:")
print(data.head())
print("\nUnique labels:", data['annotations'].unique())
print("Number of unique labels:", data['annotations'].nunique())

# Split dataset into training (70%), validation (15%), and test (15%)
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"\nTraining set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

Dataset columns: ['image_id', 'image_name', 'annotations']

First few rows:
   image_id     image_name                                        annotations
0         0   images/0.png  [{'type': 'box', 'label': 'cow', 'points': [(1...
1         1   images/1.png  [{'type': 'box', 'label': 'cow', 'points': [(1...
2        10  images/10.png  [{'type': 'box', 'label': 'cow', 'points': [(6...
3        11  images/11.png  [{'type': 'box', 'label': 'cow', 'points': [(1...
4        14  images/14.png  [{'type': 'box', 'label': 'cow', 'points': [(2...

Unique labels: ["[{'type': 'box', 'label': 'cow', 'points': [(143.44, 205.25), (359.59, 398.85)]}, {'type': 'box', 'label': 'cow', 'points': [(707.55, 98.33), (766.94, 211.18)]}, {'type': 'box', 'label': 'cow', 'points': [(545.02, 128.0), (610.4, 260.31)]}, {'type': 'box', 'label': 'cow', 'points': [(443.51, 77.27), (579.04, 164.74)]}, {'type': 'box', 'label': 'cow', 'points': [(245.17, 77.92), (326.2, 157.56)]}, {'type': 'box', 'label': 'cow', 'point

In [2]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Determine the actual number of unique labels
unique_labels = sorted(data['annotations'].unique())
num_labels = len(unique_labels)
print(f"\nNumber of classes: {num_labels}")

# Create label mapping (map labels to 0, 1, 2, ..., num_labels-1)
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
print(f"Label mapping: {label_to_id}")

# Load pre-trained BERT model with correct number of labels
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

# Step 1: Freeze all layers except the last one (classification head)
for param in model.base_model.parameters():
    param.requires_grad = False

# If you'd like to fine-tune additional layers (e.g., the last 2 layers), you can unfreeze those layers as well
for param in model.base_model.encoder.layer[-2:].parameters():
    param.requires_grad = True

# Create a proper PyTorch Dataset class
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    
    def __len__(self):
        return len(self.labels)

# Convert your pandas DataFrames to proper datasets
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Map labels to sequential IDs (0, 1, 2, ...)
train_labels = [label_to_id[label] for label in train_data['annotations']]
val_labels = [label_to_id[label] for label in val_data['annotations']]
test_labels = [label_to_id[label] for label in test_data['annotations']]  # Create test labels

print(f"\nLabel ranges:")
print(f"Train labels range: {min(train_labels)} to {max(train_labels)}")
print(f"Val labels range: {min(val_labels)} to {max(val_labels)}")
print(f"Test labels range: {min(test_labels)} to {max(test_labels)}")

# Tokenize your data (using image_id as text - not ideal but will work)
train_encodings = tokenizer(
    train_data['image_id'].astype(str).tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

val_encodings = tokenizer(
    val_data['image_id'].astype(str).tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

test_encodings = tokenizer(
    test_data['image_id'].astype(str).tolist(),  # Tokenize test data
    truncation=True,
    padding=True,
    max_length=512
)

# Create PyTorch datasets
train_dataset = TextDataset(train_encodings, train_labels)
val_dataset = TextDataset(val_encodings, val_labels)
test_dataset = TextDataset(test_encodings, test_labels)  # Create test dataset

print(f"\nDataset sizes:")
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")

# Define metric computation function
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 score."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Step 2: Set training arguments for fine-tuning the model
training_args = TrainingArguments(
    output_dir='./results',             # Directory where results will be stored
    num_train_epochs=3,                 # Number of epochs (full passes through the dataset)
    per_device_train_batch_size=16,     # Batch size per GPU/CPU during training
    per_device_eval_batch_size=16,      # Batch size for evaluation
    eval_strategy="epoch",              # Evaluate the model at the end of each epoch
    save_strategy="epoch",              # Save checkpoint at the end of each epoch
    load_best_model_at_end=True,        # Load the best model at the end of training
    metric_for_best_model="accuracy",   # Use accuracy to determine the best model
    report_to="none",                   # Disable automatic logging to MLflow to avoid parameter limit
    logging_steps=10,
    logging_dir='./logs',               # Directory for storing logs
)

# Step 3: Fine-tune only the final classification head (since earlier layers were frozen)
trainer = Trainer(
    model=model,                        # Pre-trained BERT model with frozen layers
    args=training_args,                 # Training arguments
    train_dataset=train_dataset,        # Use the converted dataset
    eval_dataset=val_dataset,           # Use the converted dataset
    compute_metrics=compute_metrics,    # Function to compute metrics
)

# Step 4: Train the model using PEFT (this performs PEFT because layers were frozen in Step 1)
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)

# Step 5: Evaluate on validation set
print("\n" + "="*80)
print("VALIDATION SET EVALUATION")
print("="*80)
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(f"Validation Results:")
for key, value in val_results.items():
    print(f"  {key}: {value:.4f}")

# Step 6: Evaluate on test set
print("\n" + "="*80)
print("TEST SET EVALUATION")
print("="*80)
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(f"\nTest Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

# Step 7: Make predictions on test set
print("\n" + "="*80)
print("PREDICTIONS ON TEST SET")
print("="*80)
predictions = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Convert predicted IDs back to original labels
predicted_original_labels = [id_to_label[pred] for pred in predicted_labels]
true_original_labels = [id_to_label[label] for label in test_labels]

# Show first 10 predictions
print("\nFirst 10 predictions vs actual:")
print(f"{'Image ID':<20} {'Predicted':<15} {'Actual':<15} {'Correct':<10}")
print("-" * 60)
for i in range(min(10, len(test_data))):
    img_id = test_data.iloc[i]['image_id']
    pred = predicted_original_labels[i]
    actual = true_original_labels[i]
    correct = "✓" if pred == actual else "✗"
    print(f"{str(img_id):<20} {str(pred):<15} {str(actual):<15} {correct:<10}")

# Save predictions to file
import pandas as pd
predictions_df = pd.DataFrame({
    'image_id': test_data['image_id'].values,
    'predicted_label': predicted_original_labels,
    'actual_label': true_original_labels,
    'correct': [pred == actual for pred, actual in zip(predicted_original_labels, true_original_labels)]
})
predictions_df.to_csv('./results/test_predictions.csv', index=False)
print(f"\n✓ Predictions saved to './results/test_predictions.csv'")

# Print final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"Best Model Performance:")
print(f"  Validation Accuracy: {val_results.get('eval_accuracy', 0):.4f}")
print(f"  Test Accuracy: {test_results.get('eval_accuracy', 0):.4f}")
print(f"  Test Precision: {test_results.get('eval_precision', 0):.4f}")
print(f"  Test Recall: {test_results.get('eval_recall', 0):.4f}")
print(f"  Test F1-Score: {test_results.get('eval_f1', 0):.4f}")

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-27 05:01:20.922917: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-27 05:01:20.985887: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-27 05:01:22.770052: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may s


Number of classes: 51
Label mapping: {"[{'type': 'box', 'label': 'cow', 'points': [(0.0, 289.54), (236.69, 431.61)]}, {'type': 'box', 'label': 'cow', 'points': [(21.32, 190.78), (122.09, 263.98)]}, {'type': 'box', 'label': 'cow', 'points': [(0.0, 243.22), (52.46, 349.0)]}, {'type': 'box', 'label': 'cow', 'points': [(173.9, 150.25), (313.9, 203.73)]}, {'type': 'box', 'label': 'cow', 'points': [(264.48, 94.02), (421.4, 216.35)]}, {'type': 'box', 'label': 'cow', 'points': [(441.2, 168.12), (558.52, 231.29)]}, {'type': 'box', 'label': 'cow', 'points': [(478.8, 195.19), (662.8, 278.92)]}, {'type': 'box', 'label': 'cow', 'points': [(573.56, 175.14), (686.36, 239.31)]}, {'type': 'box', 'label': 'cow', 'points': [(657.03, 130.62), (842.53, 312.1)]}, {'type': 'box', 'label': 'cow', 'points': [(719.6, 202.24), (833.1, 267.0)]}, {'type': 'box', 'label': 'cow', 'points': [(873.01, 58.47), (940.47, 108.76)]}, {'type': 'box', 'label': 'cow', 'points': [(564.36, 106.98), (624.94, 139.01)]}, {'type':

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Label ranges:
Train labels range: 2 to 50
Val labels range: 11 to 49
Test labels range: 0 to 39

Dataset sizes:
Train dataset: 35 samples
Validation dataset: 8 samples
Test dataset: 8 samples


/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



STARTING TRAINING


... (6 outputs left out)

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Test Results:
  eval_loss: 4.0404
  eval_accuracy: 0.0000
  eval_precision: 0.0000
  eval_recall: 0.0000
  eval_f1: 0.0000
  eval_runtime: 0.0767
  eval_samples_per_second: 104.2740
  eval_steps_per_second: 13.0340
  epoch: 3.0000

PREDICTIONS ON TEST SET


/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/sklearn/metr


First 10 predictions vs actual:
Image ID             Predicted       Actual          Correct   
------------------------------------------------------------
78                   [{'type': 'box', 'label': 'cow', 'points': [(667.71, 373.36), (774.23, 408.85)]}, {'type': 'box', 'label': 'cow', 'points': [(494.57, 445.79), (635.97, 524.72)]}, {'type': 'box', 'label': 'cow', 'points': [(559.25, 415.12), (779.1, 638.3)]}, {'type': 'box', 'label': 'cow', 'points': [(831.65, 350.52), (1008.0, 464.1)]}, {'type': 'box', 'label': 'cow', 'points': [(837.25, 376.32), (1008.0, 488.47)]}, {'type': 'box', 'label': 'cow', 'points': [(882.02, 377.55), (1008.0, 520.46)]}, {'type': 'box', 'label': 'cow', 'points': [(848.13, 431.21), (1008.0, 622.4)]}, {'type': 'box', 'label': 'cow', 'points': [(591.42, 327.06), (656.65, 378.79)]}, {'type': 'box', 'label': 'cow', 'points': [(604.88, 309.98), (653.9, 352.98)]}, {'type': 'box', 'label': 'cow', 'points': [(678.06, 303.78), (712.98, 350.11)]}, {'type': 'box',